<a href="https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/uomna/flyrank-ml/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

Feature vector: 21 numeric columns knowable before the label window, built
from the same starter dataset used across this capstone
(content_refresh_anonymized.csv, filtered to 28,795 rows with a valid
average position). Missing values are median-imputed; features are then
scaled with StandardScaler for any distance- or coefficient-based model.
Two categorical fields (content_type, main_intent) exist in the raw data
but are kept as context only, not encoded into the model — see Section 4
for why.

In [6]:
!git clone https://github.com/uomna/flyrank-ml.git repo
%cd repo
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
df_valid = df[df["avg_position"] > 0].copy()

# Numeric feature set: knowable before the label window, excludes
# id columns, availability flags, and anything that defines the label
feature_cols = [
    "search_volume", "competition", "cpc", "word_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d",
    "scroll_events_90d", "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct"
]

X = df_valid[feature_cols]

# Categorical handling: content_type and main_intent are descriptive
# context, not used as model inputs in this lane (see Section 4 for why)

# Fill missing numeric values with the median (fit on this data; in the
# actual train/test pipeline this is fit on train only, see ML-08)
imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(imputer.fit_transform(X), columns=feature_cols)

# Scale for any distance/coefficient-based model
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X_imputed), columns=feature_cols)

print("Feature vector shape:", X_scaled.shape)
print("\nMissing values before imputation:")
print(X.isna().sum()[X.isna().sum() > 0])

Cloning into 'repo'...
remote: Enumerating objects: 198, done.
remote: Counting objects: 100% (198/198), done.
remote: Compressing objects: 100% (154/154), done.
remote: Total 198 (delta 88), reused 92 (delta 27), pack-reused 0 (from 0)
Receiving objects: 100% (198/198), 1.96 MiB | 13.85 MiB/s, done.
Resolving deltas: 100% (88/88), done.
/content/repo/repo
Feature vector shape: (28795, 21)

Missing values before imputation:
search_volume    1719
competition      1719
cpc              1719
word_count       7686
scroll_rate       121
dtype: int64


## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing? | Categorical? | Available before label? |
|---|---|---|---|---|
| search_volume | Monthly search demand for the target keyword | 6.0% (1,719) | No | Yes — set at content creation |
| competition | Keyword competition score (0-1) | 6.0% (1,719) | No | Yes |
| cpc | Cost-per-click for the keyword | 6.0% (1,719) | No | Yes |
| word_count | Content length | 26.7% (7,686) | No | Yes |
| impressions_90d, clicks_90d, pageviews_90d, sessions_90d, users_90d, engaged_sessions_90d, ai_sessions_90d, scroll_events_90d | 90-day trailing traffic metrics | 0% | No | Yes — trailing window, computed before the label moment |
| days_with_impressions, days_with_sessions | Consistency of visibility/engagement over 90 days | 0% | No | Yes |
| content_age_days | Days since the content was published | 0% | No | Yes |
| days_since_last_update | Days since the content was last edited | 0% | No | Yes |
| ctr, avg_position | Click-through rate and average search position | 0% | No | Yes |
| engagement_rate, scroll_rate, ai_traffic_pct | Behavioral engagement signals | scroll_rate: 0.4% (121) | No | Yes |
| content_type, main_intent | Content category and search intent | 0% | Yes | Context only — not used as a model input (see Section 4) |

Missing values are not random for search_volume/competition/cpc (they
travel together at exactly 6.0% — likely rows where the keyword-research
step was skipped) and for word_count (26.7% — likely non-keyword-article
content types where word count wasn't tracked the same way). Both are
imputed by median in the model pipeline rather than dropped, since
dropping would lose over a quarter of rows for word_count alone.

In [7]:
# Confirm search_volume/competition/cpc are missing together (not independently random)
together = df_valid[["search_volume", "competition", "cpc"]].isna().all(axis=1).sum()
any_missing = df_valid[["search_volume", "competition", "cpc"]].isna().any(axis=1).sum()
print(f"Rows missing all three (search_volume, competition, cpc): {together}")
print(f"Rows missing at least one of the three: {any_missing}")
print(f"Same count confirms they travel together: {together == any_missing}")

# Check if word_count missingness correlates with content_type
print("\nword_count missing rate by content_type:")
print(df_valid.groupby("content_type")["word_count"].apply(lambda x: x.isna().mean()))

Rows missing all three (search_volume, competition, cpc): 1719
Rows missing at least one of the three: 1719
Same count confirms they travel together: True

word_count missing rate by content_type:
content_type
comparison article    0.000000
feedly article        0.000000
keyword article       0.287521
Name: word_count, dtype: float64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

Attacking the feature vector for label-derived columns, future windows,
and product flags:

1. Label-derived columns: is_declining_label is built from
   trend_direction/trend_pct. Neither is in feature_cols — confirmed by
   direct membership check below.
2. Near-siblings of the label: a first model attempt using
   impressions_last_30d/impressions_prev_30d (and their clicks/sessions
   counterparts) scored a suspicious 1.00 precision@50, with those two
   columns carrying coefficients 10-30x every other feature. These are
   the raw components trend_pct was likely computed from. The entire
   last_30d/prev_30d family (6 columns) is excluded from feature_cols.
3. Future windows: this starter CSV is a single snapshot with no
   per-row date, so there is no explicit future window to leak from —
   but this also means no feature can be verified as "before" the label
   in a time-stamped sense, only in a logical sense (see Section 4).
4. Product/availability flags: client_has_gsc, client_has_ga4,
   gsc_data_available, ga4_data_available are excluded — not because
   they leak the label, but because they encode measurement
   availability, not content performance, and including them risks the
   model learning "we have GSC data" as a proxy for something else.
5. ID columns: content_id and client_id are excluded from feature_cols
   — used only for joining/grouping, confirmed below.

Test result: the leakage attack (attempt 1, with last_30d/prev_30d
included) scored 1.00 precision@50. The clean feature set (attempt 2,
this notebook's feature_cols) scores 0.74 on a grouped split — an
18-point observed gap versus the rule baseline, with no single feature
coefficient dominating the way the leaky ones did.

In [8]:
# 1 & 5. Confirm label columns and id columns are not in the feature set
forbidden_label_cols = ["trend_direction", "trend_pct", "is_declining_label"]
forbidden_id_cols = ["content_id", "client_id"]

leak_found = [c for c in forbidden_label_cols if c in feature_cols]
id_found = [c for c in forbidden_id_cols if c in feature_cols]
print("Label columns in feature set (must be empty):", leak_found)
print("ID columns in feature set (must be empty):", id_found)

# 2. Reproduce the leakage attack: add the suspect family back in and retrain
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression

leaky_family = ["impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
                "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d"]
feature_cols_leaky = feature_cols + leaky_family

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(df_valid, groups=df_valid["client_id"]))
train_df = df_valid.iloc[train_idx]
test_df = df_valid.iloc[test_idx].copy()

def run_and_score(cols):
    imp = SimpleImputer(strategy="median")
    sca = StandardScaler()
    X_tr = sca.fit_transform(imp.fit_transform(train_df[cols]))
    X_te = sca.transform(imp.transform(test_df[cols]))
    m = LogisticRegression(max_iter=1000, random_state=42)
    m.fit(X_tr, train_df["is_declining_label"])
    scores = m.predict_proba(X_te)[:, 1]
    top50 = test_df.assign(score=scores).sort_values("score", ascending=False).head(50)
    return top50["is_declining_label"].mean(), m

leaky_p50, _ = run_and_score(feature_cols_leaky)
clean_p50, clean_model = run_and_score(feature_cols)

print(f"\nLeakage attack (with last_30d/prev_30d family): precision@50 = {leaky_p50:.2f}")
print(f"Clean feature set (this notebook's feature_cols): precision@50 = {clean_p50:.2f}")

Label columns in feature set (must be empty): []
ID columns in feature set (must be empty): []

Leakage attack (with last_30d/prev_30d family): precision@50 = 1.00
Clean feature set (this notebook's feature_cols): precision@50 = 0.74


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

Excluded fields, with reasons:

- trend_direction, trend_pct — define the label itself
  (is_declining_label); using them as features would leak the answer
  directly.
- impressions_last_30d, clicks_last_30d, sessions_last_30d,
  impressions_prev_30d, clicks_prev_30d, sessions_prev_30d — near-siblings
  of the label; confirmed in Section 3 to produce a suspicious 1.00
  precision@50 when included.
- client_has_gsc, client_has_ga4, gsc_data_available, ga4_data_available —
  data-availability flags, not performance signals. Including them risks
  the model confusing "we don't have data for this row" with "this
  content performed poorly."
- content_id, client_id — identifiers only, used for joining and for the
  grouped train/test split, never as model inputs.
- content_type, main_intent — descriptive context (used to interpret
  results, e.g. the word_count missingness check in Section 2), but not
  one-hot encoded into the model in this lane. Keeping the model to
  numeric performance signals only keeps the coefficient table directly
  interpretable for the reason codes used downstream in the action
  playbook (ML-10).

Public-safety note: no raw client names, URLs, or search queries exist in
this dataset — content_id and client_id are already anonymized hashes at
the source.

In [9]:
excluded_fields = {
    "trend_direction": "defines the label directly",
    "trend_pct": "defines the label directly",
    "impressions_last_30d": "near-sibling of the label (leakage confirmed in Section 3)",
    "clicks_last_30d": "near-sibling of the label",
    "sessions_last_30d": "near-sibling of the label",
    "impressions_prev_30d": "near-sibling of the label",
    "clicks_prev_30d": "near-sibling of the label",
    "sessions_prev_30d": "near-sibling of the label",
    "client_has_gsc": "data-availability flag, not a performance signal",
    "client_has_ga4": "data-availability flag, not a performance signal",
    "gsc_data_available": "data-availability flag, not a performance signal",
    "ga4_data_available": "data-availability flag, not a performance signal",
    "content_id": "identifier only, used for joining/grouping",
    "client_id": "identifier only, used for joining/grouping",
    "content_type": "descriptive context, not one-hot encoded in this lane",
    "main_intent": "descriptive context, not one-hot encoded in this lane",
}

print(f"Total excluded fields: {len(excluded_fields)}\n")
for field, reason in excluded_fields.items():
    print(f"  {field}: {reason}")

# Sanity check: confirm none of these appear in the final feature_cols
overlap = [f for f in excluded_fields if f in feature_cols]
print(f"\nOverlap with feature_cols (must be empty): {overlap}")

Total excluded fields: 16

  trend_direction: defines the label directly
  trend_pct: defines the label directly
  impressions_last_30d: near-sibling of the label (leakage confirmed in Section 3)
  clicks_last_30d: near-sibling of the label
  sessions_last_30d: near-sibling of the label
  impressions_prev_30d: near-sibling of the label
  clicks_prev_30d: near-sibling of the label
  sessions_prev_30d: near-sibling of the label
  client_has_gsc: data-availability flag, not a performance signal
  client_has_ga4: data-availability flag, not a performance signal
  gsc_data_available: data-availability flag, not a performance signal
  ga4_data_available: data-availability flag, not a performance signal
  content_id: identifier only, used for joining/grouping
  client_id: identifier only, used for joining/grouping
  content_type: descriptive context, not one-hot encoded in this lane
  main_intent: descriptive context, not one-hot encoded in this lane

Overlap with feature_cols (must be empty)

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.